# Затухающие колебания

**Главная мысль занятия.** Решение уравнения $\ddot x + \gamma\dot x + \omega_0^2 x = 0$ ищем в виде $e^{\lambda t}$, и $\lambda$ — корень уравнения $\lambda^2 + \gamma\lambda + \omega_0^2 = 0$. Корень — это комплексная частота: $-\mathrm{Re}\,\lambda$ говорит, как быстро движение затухает, $\mathrm{Im}\,\lambda$ — с какой частотой груз колеблется. Трение двигает корни, а вместе с ними меняется движение.

**Как запустить.** Нажмите **Ctrl+F9** (или «Среда выполнения» → «Выполнить всё»). На предупреждение «блокнот создан не Google» ответьте «Всё равно выполнить». Через несколько секунд ниже появятся картинки с кнопками и ползунками.

Всё управление — кнопками и ползунками.

In [ ]:
#@title Вычисления { display-mode: "form" }
# =============================================================
#  Движок: осциллятор с трением x'' + γx' + ω0² x = 0 (точные
#  формулы по трём случаям) и подвеска машины. Интерфейс — в
#  следующих ячейках.
# =============================================================
import numpy as np

C_ONE, C_TWO, C_THREE = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK2, GRID, PATH, GHOST = '#0b0b0b', '#52514e', '#e6e5e0', '#cfcec7', '#b5b4ad'


def fmt(x, d=2):
    return ('%.*f' % (d, x)).replace('.', ',').replace('-', '−')


def clean(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.tick_params(colors=INK2)
    ax.grid(color=GRID, lw=0.8)


def roots(g, w0=1.0):
    """Корни λ² + γλ + ω0² = 0; первым — медленный (ближе к мнимой оси)."""
    D = g * g - 4 * w0 * w0
    if D < 0:
        wd = np.sqrt(-D) / 2
        return complex(-g / 2, wd), complex(-g / 2, -wd)
    s = np.sqrt(D) / 2
    return complex(-g / 2 + s, 0.0), complex(-g / 2 - s, 0.0)


def is_critical(g, w0=1.0):
    return abs(g * g - 4 * w0 * w0) < 1e-9 * max(1.0, w0 ** 2)


def motion(g, t, w0=1.0, x0=1.0, v0=0.0):
    """Точное решение x'' + γx' + ω0² x = 0 с x(0) = x0, x'(0) = v0."""
    D = g * g - 4 * w0 * w0
    if is_critical(g, w0):                        # кратный корень — практика 8
        l = -g / 2
        return (x0 + (v0 - l * x0) * t) * np.exp(l * t)
    if D < 0:                                     # затухающие колебания
        wd = np.sqrt(-D) / 2
        return np.exp(-g * t / 2) * (x0 * np.cos(wd * t)
                                     + (v0 + g * x0 / 2) / wd * np.sin(wd * t))
    l1, l2 = (-g + np.sqrt(D)) / 2, (-g - np.sqrt(D)) / 2
    A = (v0 - l2 * x0) / (l1 - l2)                # две экспоненты
    return A * np.exp(l1 * t) + (x0 - A) * np.exp(l2 * t)


def settle_time(g, w0=1.0, tol=0.05):
    """С какого момента |x| навсегда не больше tol от начального отклонения
    (отпустили из x0 без скорости). None — трения нет, не успокаивается."""
    if g <= 0:
        return None
    rate = -roots(g, w0)[0].real                  # медленнейшая скорость затухания
    t_max = 12.0 / rate + 4 * np.pi / w0
    tt = np.linspace(0, t_max, 400001)
    big = np.nonzero(np.abs(motion(g, tt, w0)) > tol)[0]
    return float(tt[big[-1]]) if len(big) else 0.0


def overshoot(g, w0=1.0):
    """Первый перелёт через ноль, в долях начального отклонения."""
    z = g / (2 * w0)
    return float(np.exp(-np.pi * z / np.sqrt(1 - z * z))) if z < 1 else 0.0


# ---------- подвеска: четверть машины ----------
G_ACC = 9.81


def car(m, k_kN, c):
    k = k_kN * 1e3
    w0 = np.sqrt(k / m)
    c_cr = 2 * np.sqrt(k * m)
    return dict(w0=w0, f0=w0 / (2 * np.pi), sag=m * G_ACC / k, c_cr=c_cr,
                zeta=c / c_cr, g=c / m)


---

## 1. Корни и движение

Сверху — корни характеристического уравнения на комплексной плоскости (здесь $\omega_0 = 1$). Серым нарисован путь, по которому они идут, когда трение растёт от нуля до шести. Внизу — движение груза, которого отвели в $x = 1$ и отпустили без скорости. Зелёная полоса — коридор $\pm 5\,\%$: когда груз вошёл в неё навсегда, считаем, что он успокоился.

In [ ]:
#@title Виджет: корни и движение { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

T_SHOW, G_MAX = 25.0, 6.0
TH = np.linspace(np.pi / 2, 3 * np.pi / 2, 200)
L_END = roots(G_MAX)                              # куда корни уходят при γ = 6
T_CRIT = settle_time(2.0)
PRESETS_1 = [('без трения', 0.0), ('слабое', 0.6), ('критическое', 2.0), ('сильное', 6.0)]


def roots_text(g):
    l1, l2 = roots(g)
    if g == 0:
        return 'λ = ± i'
    if is_critical(g):
        return 'λ₁ = λ₂ = −1'
    if l1.imag != 0:
        return 'λ = %s ± %s i' % (fmt(l1.real), fmt(abs(l1.imag)))
    return 'λ₁ = %s,   λ₂ = %s' % (fmt(l1.real), fmt(l2.real))


def regime_text(g):
    if g == 0:
        return ('<b>Трения нет.</b> Корни ±i лежат на мнимой оси: затухания нет, колебания '
                'с частотой ω₀ = 1 — это осциллятор из практики 6.')
    if is_critical(g):
        return ('<b>Критическое трение.</b> Корни столкнулись в −1. Колебаний уже нет, а '
                'медленный корень дальше всего от мнимой оси. Формула с двумя экспонентами '
                'здесь ломается: корень один — это практика 8.')
    if g < 2:
        return ('<b>Слабое трение.</b> Корни идут по окружности радиуса ω₀: '
                'x = e<sup>−γt/2</sup>(A cos ω<sub>d</sub>t + B sin ω<sub>d</sub>t) — '
                'колебания под огибающей. Чем больше γ, тем левее корни и тем быстрее '
                'гаснет огибающая.')
    return ('<b>Сильное трение.</b> Корни разошлись по оси: один убегает влево и быстро '
            'гаснет, другой ползёт к нулю — груз вязнет. Медленный корень ≈ −ω₀²/γ.')


class RootsApp:

    def __init__(self):
        self.w_g = widgets.FloatSlider(value=0.6, min=0.0, max=G_MAX, step=0.05,
                                       description='трение γ', readout_format='.2f',
                                       style={'description_width': '70px'},
                                       layout=widgets.Layout(width='520px'))
        self.buttons = []
        for name, g in PRESETS_1:
            b = widgets.Button(description=name, layout=widgets.Layout(width='120px'))
            b.on_click(lambda _, g=g: setattr(self.w_g, 'value', g))
            self.buttons.append(b)
        self.w_crit = widgets.Checkbox(value=True, indent=False,
                                       description='серым — критическое')
        self.out, self.info = widgets.Output(), widgets.HTML()
        self.w_g.observe(lambda ch: self._refresh(), names='value')
        self.w_crit.observe(lambda ch: self._refresh(), names='value')
        display(self.w_g, widgets.HBox(self.buttons + [self.w_crit]), self.out, self.info)
        self._refresh()

    def _refresh(self):
        g = self.w_g.value
        l1, l2 = roots(g)
        ts = settle_time(g)
        tt = np.linspace(0, T_SHOW, 1500)
        with self.out:
            clear_output(wait=True)
            fig = plt.figure(figsize=(11.2, 7.0), dpi=88)
            gs = fig.add_gridspec(2, 2, width_ratios=[1.75, 1.0], height_ratios=[1.0, 1.15],
                                  hspace=0.38, wspace=0.06)
            fig.subplots_adjust(left=0.06, right=0.98, top=0.95, bottom=0.08)
            a1, a0, a2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])

            # --- корни
            a1.plot(np.cos(TH), np.sin(TH), color=PATH, lw=3.5, zorder=1)
            a1.plot([L_END[1].real, L_END[0].real], [0, 0], color=PATH, lw=3.5, zorder=1)
            a1.axvline(0, color=INK2, lw=1.0)
            a1.axhline(0, color=INK2, lw=0.6)
            if g > 0:
                a1.axvline(l1.real, color=C_TWO, lw=1.4, ls='--', zorder=2)
            a1.plot([l1.real, l2.real], [l1.imag, l2.imag], 'o', ms=13, color=C_ONE,
                    mec='white', mew=2, zorder=5)
            a1.set_xlim(-6.2, 0.6)
            a1.set_ylim(-1.45, 1.45)
            a1.set_aspect('equal')
            a1.set_xlabel('Re λ   (левее — быстрее затухает)')
            a1.set_ylabel('Im λ — частота')
            a1.set_title('корни λ² + γλ + 1 = 0', loc='left', color=INK, fontsize=11)
            clean(a1)

            # --- числа
            a0.axis('off')
            if g == 0:
                rate, freq = 'нет', '1'
            else:
                rate = fmt(-l1.real)
                freq = fmt(abs(l1.imag)) + '  (без трения — 1)' if l1.imag != 0 else 'колебаний нет'
            if ts is None:
                st = 'не успокаивается'
            elif ts > T_SHOW:
                st = 't ≈ %s (за краем графика)' % fmt(ts, 0)
            else:
                st = 't = ' + fmt(ts, 1)
            rows = [('корни', roots_text(g)),
                    ('затухание', rate),
                    ('частота', freq),
                    ('успокоился', st),
                    ('критическое', 't = ' + fmt(T_CRIT, 1))]
            for i, (k, v) in enumerate(rows):
                y = 0.92 - 0.19 * i
                a0.text(0.03, y, k, transform=a0.transAxes, fontsize=10.5, color=INK2, va='top')
                a0.text(0.03, y - 0.075, v, transform=a0.transAxes, fontsize=13,
                        color=C_TWO if k == 'успокоился' else INK, va='top')

            # --- движение
            a2.axhspan(-0.05, 0.05, color=C_THREE, alpha=0.18, lw=0)
            if self.w_crit.value and not is_critical(g):
                a2.plot(tt, motion(2.0, tt), color=GHOST, lw=1.6, ls='--', label='критическое')
            if g < 2 and not is_critical(g):
                env = np.exp(-g * tt / 2)
                a2.plot(tt, env, color=C_ONE, lw=1, alpha=0.45)
                a2.plot(tt, -env, color=C_ONE, lw=1, alpha=0.45)
            a2.plot(tt, motion(g, tt), color=C_ONE, lw=2.5, label='γ = ' + fmt(g))
            if ts is not None and ts <= T_SHOW:
                a2.axvline(ts, color=C_TWO, lw=1.4, ls='--')
                right = ts > 0.75 * T_SHOW
                a2.text(ts + (-0.3 if right else 0.3), 0.8, 'успокоился', color=C_TWO,
                        fontsize=10, ha='right' if right else 'left')
            a2.set_xlim(0, T_SHOW)
            a2.set_ylim(-1.05, 1.05)
            a2.set_xlabel('t')
            a2.set_title('x(t): отпустили из x = 1 без скорости; зелёная полоса — ±5 %',
                         loc='left', color=INK, fontsize=11)
            a2.legend(loc='upper right', fontsize=9, frameon=True, facecolor='white', edgecolor='none', framealpha=0.92)
            clean(a2)
            plt.show()
        self.info.value = "<div style='font-size:14px'>%s</div>" % regime_text(g)


roots_app = RootsApp()


### Что попробовать

1. **Сначала прогноз.** Какое трение вернёт груз в покой быстрее всего: слабое, очень сильное или какое-то среднее? Запишите ответ, потом двигайте ползунок и смотрите на строку «успокоился».

2. **Ведите трение от 0 до 2.** Точки едут по серой окружности навстречу друг другу. Почему именно по окружности радиуса $\omega_0 = 1$? Подсказка: чему равно произведение корней?

3. **Дальше 2.** Корни столкнулись и разошлись по оси. Какой из двух решает, сколько груз будет ползти к нулю? Что с ним стало при $\gamma = 6$?

4. **Охота за рекордом.** Найдите трение, при котором груз успокаивается быстрее всего. Это критическое $\gamma = 2$? Посмотрите, какого размера первый перелёт через ноль, когда время минимально.

---

## 2. Подвеска после кочки

Четверть легковой машины: на одно колесо приходится около 300 кг кузова, пружина — около 30 кН/м. Кочка подбросила кузов на 5 см, дальше он движется сам: $m\ddot x = -kx - c\dot x$, где $c$ — сопротивление амортизатора (Н·с/м). Кнопки ставят амортизатор в долях от критического.

In [ ]:
#@title Виджет: подвеска после кочки { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

T_CAR, X0_CM = 4.0, 5.0
PRESETS_2 = [('обычный (0,3 от критического)', 0.3), ('критический', 1.0),
             ('сломан (0,05)', 0.05)]


class CarApp:

    def __init__(self):
        sl = dict(style={'description_width': '170px'}, layout=widgets.Layout(width='480px'))
        self.w_m = widgets.FloatSlider(value=300, min=150, max=500, step=10,
                                       description='масса на колесо, кг',
                                       readout_format='.0f', **sl)
        self.w_k = widgets.FloatSlider(value=30, min=10, max=60, step=1,
                                       description='пружина, кН/м', readout_format='.0f', **sl)
        self.w_c = widgets.FloatSlider(value=1800, min=0, max=12000, step=100,
                                       description='амортизатор, Н·с/м',
                                       readout_format='.0f', **sl)
        self.buttons = []
        for name, z in PRESETS_2:
            b = widgets.Button(description=name, layout=widgets.Layout(width='230px'))
            b.on_click(lambda _, z=z: self._set_zeta(z))
            self.buttons.append(b)
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in (self.w_m, self.w_k, self.w_c):
            w.observe(lambda ch: self._refresh(), names='value')
        display(widgets.VBox([self.w_m, self.w_k, self.w_c]), widgets.HBox(self.buttons),
                self.out, self.info)
        self._refresh()

    def _set_zeta(self, z):
        c_cr = car(self.w_m.value, self.w_k.value, 0.0)['c_cr']
        self.w_c.value = float(np.clip(round(z * c_cr / 100) * 100, 0, self.w_c.max))

    def _refresh(self):
        m, k, c = self.w_m.value, self.w_k.value, self.w_c.value
        p = car(m, k, c)
        tt = np.linspace(0, T_CAR, 1500)
        x = X0_CM * motion(p['g'], tt, p['w0'])
        ts = settle_time(p['g'], p['w0'])
        ts_cr = settle_time(2 * p['w0'], p['w0'])
        with self.out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10.4, 4.4), dpi=88)
            fig.subplots_adjust(left=0.08, right=0.98, top=0.9, bottom=0.13)
            ax.axhspan(-0.05 * X0_CM, 0.05 * X0_CM, color=C_THREE, alpha=0.18, lw=0)
            ax.plot(tt, X0_CM * motion(2 * p['w0'], tt, p['w0']), color=GHOST, lw=1.6,
                    ls='--', label='критический амортизатор')
            ax.plot(tt, x, color=C_ONE, lw=2.6, label='ваш амортизатор')
            if ts is not None and ts <= T_CAR:
                ax.axvline(ts, color=C_TWO, lw=1.4, ls='--')
            ax.set_xlim(0, T_CAR)
            ax.set_ylim(-1.05 * X0_CM, 1.05 * X0_CM)
            ax.set_xlabel('время, с')
            ax.set_ylabel('кузов, см')
            ax.set_title('кочка подбросила кузов на 5 см; зелёная полоса — ±5 %',
                         loc='left', color=INK, fontsize=11)
            ax.legend(loc='upper right', fontsize=9, frameon=True, facecolor='white', edgecolor='none', framealpha=0.92)
            clean(ax)
            plt.show()
        if ts is None:
            st = 'не успокаивается совсем — амортизатора нет'
        elif ts > T_CAR:
            st = 'успокоится через %s с — за краем графика' % fmt(ts, 1)
        else:
            st = 'успокоится через <b>%s с</b>' % fmt(ts, 2)
        ov = overshoot(p['g'], p['w0'])
        ov_txt = ('первый перелёт вниз — <b>%s %%</b> (%s см)' % (fmt(100 * ov, 0), fmt(ov * X0_CM, 1))
                  if ov > 0 else 'перелёта нет')
        self.info.value = (
            "<div style='font-size:14px; line-height:1.6'>"
            "Собственная частота f₀ = <b>%s Гц</b> (период %s с). Под весом пружина сжата на %s см.<br>"
            "Критический амортизатор 2√(km) = %s Н·с/м; ваш — <b>%s</b> от него.<br>"
            "Кузов: %s; %s. С критическим — %s с.</div>"
            % (fmt(p['f0'], 2), fmt(1 / p['f0'], 2), fmt(100 * p['sag'], 1), fmt(p['c_cr'], 0),
               fmt(p['zeta'], 2), ov_txt, st, fmt(ts_cr, 2)))


car_app = CarApp()


### Что попробовать

1. **«Обычный».** Сколько раз кузов качнётся, прежде чем успокоиться? Сравните с критическим: на сколько дольше?

2. **«Сломан».** Так ведёт себя машина с изношенным амортизатором. Сколько секунд кузов болтается? Это и проверяют, когда нажимают на угол машины и отпускают.

3. **Сели пассажиры:** масса на колесо 400 кг, пружина и амортизатор те же. Что стало с частотой, с просадкой и с долей от критического?

4. Почему на машины ставят амортизатор слабее критического? Ответ — на практике 8: дорога толкает колесо не один раз, а всё время.

---

## Вопросы к виджетам

1. **Почему при слабом трении корни идут по окружности?**

<details><summary>Ответ</summary>

По теореме Виета $\lambda_1\lambda_2 = \omega_0^2$. При слабом трении корни комплексно сопряжены, и их произведение равно $|\lambda|^2$. Значит, $|\lambda| = \omega_0$ при любом $\gamma < 2\omega_0$ — корни остаются на окружности радиуса $\omega_0$, меняется только угол.

</details>

2. **Почему сильное трение успокаивает медленнее критического?**

<details><summary>Ответ</summary>

Время успокоения задаёт медленный корень — ближайший к мнимой оси. При сильном трении он ползёт к нулю: $\lambda \approx -\omega_0^2/\gamma$ (при $\gamma = 6$ это $-0{,}17$ против $-1$ у критического). Физически: при большом трении инерция не важна, $c\dot x \approx -kx$ — уравнение первого порядка, как дом из практики 4. Пружина продавливает груз сквозь вязкую среду за время порядка $c/k$, и чем больше трение, тем дольше.

</details>

3. **Какое трение успокаивает быстрее всех и почему не критическое?**

<details><summary>Ответ</summary>

Около $\gamma \approx 1{,}4$ (точнее $1{,}38$): груз входит в коридор $\pm 5\,\%$ за $t \approx 2{,}9$, а при критическом — за $4{,}7$. Секрет — в первом перелёте: при этом трении он ровно $5\,\%$, груз проскакивает ноль, но из коридора уже не выходит. Если коридор сужать, рекордное трение приближается к критическому. Критическое — самое быстрое успокоение **без перелёта**.

</details>

4. **При $\gamma = 0{,}6$ во сколько раз период длиннее, чем без трения? Во сколько раз за период уменьшается размах?**

<details><summary>Ответ</summary>

Частота $\omega_d = \sqrt{1 - 0{,}3^2} \approx 0{,}954$ — период длиннее всего на $4{,}8\,\%$. А размах за период уменьшается в $e^{\gamma T_d/2} \approx 7{,}2$ раза. Трение почти не меняет частоту, но сильно гасит амплитуду.

</details>

5. **В машину сели пассажиры. Почему подвеска стала «мягче» и сильнее раскачивается?**

<details><summary>Ответ</summary>

Доля от критического $\zeta = c/(2\sqrt{km})$ падает с ростом массы: при 400 кг вместо 300 кг она $0{,}26$ вместо $0{,}30$, и первый перелёт растёт. Частота $\sqrt{k/m}/2\pi$ тоже падает, а просадка $mg/k$ растёт — с 9,8 до 13,1 см.

</details>